# Hai nhánh mô học A/B — Colab

Default chỉ preprocess, chưa train classifier. CPU modes: preprocess/cpu. T4 modes: gpu_smoke chạy hai lần độc lập rồi neural. Labels cấp ca; không gán cluster/noise/novelty thành nhãn ung thư patch. Dữ liệu nguồn giữ nguyên.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/histology")


In [ ]:
RUN_MODE="preprocess"
STUDY_ID="dual-20261011-v001"
WORK_MINUTES=90
RUNTIME_SHA256='ec9dc8bd03da9e25362a8348573bef21ed107f6a55d6c72454c8d39e82788cba'
RUNTIME_ZIP=ROOT/"runtime/histology-dual-runtime.zip"


In [ ]:
import hashlib
import json
import subprocess
import sys
import time
import zipfile
from datetime import datetime, timedelta, timezone

if hashlib.sha256(RUNTIME_ZIP.read_bytes()).hexdigest()!=RUNTIME_SHA256:raise ValueError("Runtime SHA mismatch")
CODE=Path("/content/histology-dual-runtime")/RUNTIME_SHA256[:16];CODE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(RUNTIME_ZIP) as archive:
 for name in archive.namelist():
  p=Path(name)
  if p.is_absolute() or ".." in p.parts:raise ValueError("Unsafe runtime path")
 archive.extractall(CODE)
subprocess.run([sys.executable,"-m","pip","install","--quiet","scikit-learn==1.8.0"],check=True)
sys.path.insert(0,str(CODE))


In [ ]:
from histology_data.mil_case_cache import prepare_case_cache
from histology_data.mil_runtime import stage_training_inputs

STOP=datetime.now(timezone.utc)+timedelta(minutes=WORK_MINUTES)
def deadline():return time.monotonic()+max(0,(STOP-datetime.now(timezone.utc)).total_seconds())
BUNDLE_ID="6e412d65e429ad19c3bcec645a19d7ee06fe75e560ef67149233f14d4175b0ff"
stage=stage_training_inputs(ROOT/"bundles"/BUNDLE_ID,ROOT/"features/v001",Path("/content/dual-stage"),deadline())
cache=prepare_case_cache(stage["bundle_path"],stage["feature_root"],Path("/content/dual-stage/case-caches"),deadline())
setup={k:str(v) if isinstance(v,Path) else v for k,v in stage.items()};setup["case_cache"]=str(cache)
SETUP=Path("/content/dual-source-setup.json");SETUP.write_text(json.dumps(setup))


In [ ]:
CONFIG=Path("/content/dual-config.json")
config={"study_id":STUDY_ID,"cohort":"common16","setup_path":str(SETUP),"output_root":str(ROOT/"clustering_dual/v001"/STUDY_ID),"source_code_sha256":RUNTIME_SHA256,"runtime_archive_path":str(RUNTIME_ZIP),"recover_stale_locks":False}
CONFIG.write_text(json.dumps(config))
if RUN_MODE not in {"preprocess","cpu","gpu_smoke","neural"}:raise ValueError("Unknown explicit mode")
result=subprocess.run([sys.executable,"-m","histology_data.dual_cli","--mode",RUN_MODE,"--config",str(CONFIG),"--deadline-utc",STOP.isoformat()],cwd=CODE,check=False)
if result.returncode:raise RuntimeError("Run failed; preserve artifacts and inspect status/logs.")


## Resume và kết thúc

Giữ STUDY_ID/runtime SHA/settings để resume. Chỉ bật recover_stale_locks sau khi xác minh owner cũ đã dừng; cross-host cần evidence đúng token/host/boot/session cho từng lock. Complete của CPU là cpu_complete; B1 và báo cáo vẫn có stages riêng. Không auto-click/SSH/keep-alive giả. Sau workers dừng và receipts verified, flush Drive và dừng đúng runtime.


In [ ]:
# Chỉ chạy cell này sau khi workers đã dừng và artifacts đã được kiểm.
# from google.colab import drive
# drive.flush_and_unmount(timeout_ms=60000)
# Dừng runtime qua Colab/colab-manager; không tắt máy cá nhân.
